In [ ]:
import os

DATA_ROOT = "/kaggle/input/glaucoma-fundus-and-oc-od-masks-version-3"

images_dir = os.path.join(DATA_ROOT, "fundus_images")
disc_dir   = os.path.join(DATA_ROOT, "optic disc")
cup_dir    = os.path.join(DATA_ROOT, "optic cup")

image_files = sorted(os.listdir(images_dir))
disc_files  = set(os.listdir(disc_dir))
cup_files   = set(os.listdir(cup_dir))

missing_files = []
valid_pairs   = []

for img in image_files:
    base = os.path.splitext(img)[0]   # '00001'

    disc_mask = f"OD_{base}.jpg"
    cup_mask  = f"OC_{base}.jpg"

    if disc_mask in disc_files and cup_mask in cup_files:
        valid_pairs.append(img)
    else:
        missing_files.append(img)

print(f"Total fundus images: {len(image_files)}")
print(f"Images with both masks: {len(valid_pairs)}")
print(f"Images missing one or both masks: {len(missing_files)}")

print("Example valid pairs:", valid_pairs[:5])
print("Example missing:", missing_files[:5])

In [ ]:

!pip install segmentation-models-pytorch

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import torch.nn.functional as F


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE   = 512
LR         = 1e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 8

NUM_WORKERS = 2

In [ ]:
class GlaucomaDataset(Dataset):
    def __init__(self, img_dir, disc_dir, cup_dir, transform=None):
        self.img_dir = img_dir
        self.disc_dir = disc_dir
        self.cup_dir = cup_dir
        self.transform = transform
        self.images = []

        # 1. Iterate through fundus images
        for f in sorted(os.listdir(img_dir)):
            if not f.endswith(('.jpg', '.png', '.jpeg')):
                continue

            base = os.path.splitext(f)[0] # e.g., "00002"

            # 2. Construct mask paths based on your screenshots
            disc_path = os.path.join(disc_dir, f"OD_{base}.jpg")
            cup_path  = os.path.join(cup_dir,  f"OC_{base}.jpg")

            # 3. Only add if both masks exist
            if os.path.exists(disc_path) and os.path.exists(cup_path):
                self.images.append(f)
        
        print(f"✅ Dataset loaded successfully with {len(self.images)} samples.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        base = os.path.splitext(img_name)[0]

        # Load Fundus Image
        image = cv2.imread(os.path.join(self.img_dir, img_name))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Load Masks using the OD_ and OC_ prefixes
        disc_mask = cv2.imread(os.path.join(self.disc_dir, f"OD_{base}.jpg"), 0)
        cup_mask  = cv2.imread(os.path.join(self.cup_dir,  f"OC_{base}.jpg"), 0)

        # Create multi-class mask
        # Background: 0, Disc: 1, Cup: 2
        mask = np.zeros(disc_mask.shape, dtype=np.uint8)
        mask[disc_mask > 127] = 1
        mask[cup_mask > 127]  = 2 

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]

        return image, mask.long()

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


In [ ]:
# ===============================
# 5. CUSTOM U-NET (Exact Paper Architecture)
# ===============================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


In [ ]:
class UNet(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()

        # Encoder (32 → 64 → 128)
        self.enc1 = DoubleConv(3, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(0.3)

        # Decoder (128 → 64 → 32)
        self.up2  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1  = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        # Output
        self.out_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e3 = self.dropout(e3)

        d2 = self.up2(e3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out_conv(d1)  # logits


In [ ]:
from torch.utils.data import DataLoader, random_split


img_dir = f"/kaggle/input/glaucoma-fundus-and-oc-od-masks-version-3/fundus_images"
disc_dir   = f"/kaggle/input/glaucoma-fundus-and-oc-od-masks-version-3/optic disc"
cup_dir    = f"/kaggle/input/glaucoma-fundus-and-oc-od-masks-version-3/optic cup"

dataset = GlaucomaDataset(
    img_dir, disc_dir, cup_dir,
    transform=train_transform
)

val_ratio = 0.2
val_size  = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size]
)

# Apply validation transforms
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform   = val_transform

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
print(len(train_dataset))
print(len(val_dataset))

In [ ]:
dataset = GlaucomaDataset(img_dir, disc_dir, cup_dir)
print("Number of images:", len(dataset))
print("Some sample images:", dataset.images[:5])

In [ ]:
model = UNet(n_classes=3).to(DEVICE)



In [ ]:
def validate(model, loader, device, epoch=None):
    model.eval()
    val_loss = 0.0
    dice_disc, dice_cup = [], []

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks  = masks.to(device)

            outputs = model(images)

            # Pass epoch if your loss expects it
            if epoch is not None:
                loss = criterion(outputs, masks, epoch)
            else:
                loss = criterion(outputs, masks)  # works if epoch is optional in loss

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            masks = masks.cpu().numpy()

            for i in range(preds.shape[0]):
                dice_disc.append(dice_score(preds[i], masks[i], cls=1))
                dice_cup.append(dice_score(preds[i], masks[i], cls=2))

    return (
        val_loss / len(loader),
        np.mean(dice_disc),
        np.mean(dice_cup)
    )


In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "dice_disc": [],
    "dice_cup": []
}


In [ ]:
def dice_score(pred, target, cls):
    """
    pred   : HxW predicted mask (numpy)
    target : HxW ground truth mask (numpy)
    cls    : class index (1=Disc, 2=Cup)
    """
    pred_bin   = (pred == cls).astype(np.uint8)
    target_bin = (target == cls).astype(np.uint8)

    intersection = np.sum(pred_bin * target_bin)
    union = np.sum(pred_bin) + np.sum(target_bin)

    if union == 0:
        return 1.0  # Perfect overlap if both empty

    return (2.0 * intersection) / union


In [ ]:
class ProgressiveHybridLoss(nn.Module):
    def __init__(self):
        super().__init__()

        self.dice = smp.losses.DiceLoss(
            mode="multiclass",
            classes=[1, 2],
            from_logits=True,
            log_loss=True
        )

        self.tversky = smp.losses.TverskyLoss(
            mode="multiclass",
            classes=[1, 2],
            from_logits=True,
            alpha=0.8,
            beta=0.2,
            log_loss=True
        )

        self.focal = smp.losses.FocalLoss(
            mode="multiclass"
        )

    def forward(self, preds, targets, epoch):
        # Phase 1: stabilize localization
        if epoch < 15:
            return self.dice(preds, targets)

        # Phase 2: boundary refinement
        elif epoch < 40:
            return (
                0.7 * self.tversky(preds, targets) +
                0.3 * self.focal(preds, targets)
            )

        # Phase 3: fine polish
        else:
            return (
                0.9 * self.tversky(preds, targets) +
                0.1 * self.focal(preds, targets)
            )


In [ ]:
criterion = ProgressiveHybridLoss()



optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:
print("Starting Training...")

best_dice = 0.0
EPOCHS = 50
start_epoch = 0

for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss = 0.0

    for images, masks in train_loader:
        images = images.to(DEVICE)
        masks  = masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, masks,epoch)  
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    val_loss, dice_d, dice_c = validate(model, val_loader, DEVICE,epoch)

    mean_dice = (dice_d + dice_c) / 2.0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["dice_disc"].append(dice_d)
    history["dice_cup"].append(dice_c)

    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save(
            model.state_dict(),
            f"best_ep{epoch}_D{dice_d:.3f}_C{dice_c:.3f}.pth"
        )
        print(f"--> Best Dice model saved at epoch {epoch+1}!")

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Dice(D): {dice_d:.3f} | "
        f"Dice(C): {dice_c:.3f} | "
        f"Mean Dice: {mean_dice:.3f}"
    )


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6, disc_weight=0.6, cup_weight=0.4):
        super().__init__()
        self.smooth = smooth
        self.dw = disc_weight
        self.cw = cup_weight

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)

        # Disc = class 1
        pred_d = probs[:, 1]
        gt_d   = (targets == 1).float()

        inter_d = (pred_d * gt_d).sum()
        union_d = pred_d.sum() + gt_d.sum()
        dice_d  = (2 * inter_d + self.smooth) / (union_d + self.smooth)

        # Cup = class 2
        pred_c = probs[:, 2]
        gt_c   = (targets == 2).float()

        inter_c = (pred_c * gt_c).sum()
        union_c = pred_c.sum() + gt_c.sum()
        dice_c  = (2 * inter_c + self.smooth) / (union_c + self.smooth)

        dice = self.dw * dice_d + self.cw * dice_c
        return 1 - dice
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=[0.1, 1.5, 1.0]):
        super().__init__()
        self.gamma = gamma
        self.alpha = torch.tensor(alpha)

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits, targets,
            weight=self.alpha.to(logits.device),
            reduction="none"
        )
        pt = torch.exp(-ce)
        focal = (1 - pt) ** self.gamma * ce
        return focal.mean()
class DiceFocalLoss(nn.Module):
    def __init__(self, dice_weight=0.6, focal_weight=0.4):
        super().__init__()
        self.dice = DiceLoss(disc_weight=0.6, cup_weight=0.4)
        self.focal = FocalLoss()
        self.dw = dice_weight
        self.fw = focal_weight

    def forward(self, preds, targets):
        return (
            self.dw * self.dice(preds, targets) +
            self.fw * self.focal(preds, targets)
        )
criterion = DiceFocalLoss()


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',       # higher Dice is better
    factor=0.5,
    patience=2
)

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
print("Starting Training...")

best_dice = 0.0
EPOCHS = 70
start_epoch = 49

for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss = 0.0

    for images, masks in train_loader:
        images = images.to(DEVICE)
        masks  = masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, masks)  # ✅ NO epoch
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    val_loss, dice_d, dice_c = validate(model, val_loader, DEVICE)

    mean_dice = (dice_d + dice_c) / 2.0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["dice_disc"].append(dice_d)
    history["dice_cup"].append(dice_c)

    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save(
            model.state_dict(),
            f"best_ep{epoch}_D{dice_d:.3f}_C{dice_c:.3f}.pth"
        )
        print(f"--> Best Dice model saved at epoch {epoch+1}!")

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Dice(D): {dice_d:.3f} | "
        f"Dice(C): {dice_c:.3f} | "
        f"Mean Dice: {mean_dice:.3f}"
    )


# testing

In [ ]:
# ===============================
# 10. VISUALIZATION CHECK (With Post-Processing)
# ===============================
import random

# --- 1. Define the Cleaning Function First ---
def clean_segmentation(mask_array):
    """
    Keeps only the largest connected component (the main Optic Disc).
    Removes small floating noise specs in the background.
    """
    # Create a binary mask of "Disc + Cup" vs "Background"
    binary_mask = (mask_array > 0).astype(np.uint8)

    # Find all connected blobs
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)

    # If we have more than 1 blob (background is usually label 0)
    if num_labels > 2:
        # stats columns: [left, top, width, height, area]
        # We skip index 0 because that is the background
        areas = stats[1:, 4]
        max_label = np.argmax(areas) + 1

        # Create a new clean mask
        clean_mask = np.zeros_like(mask_array)

        # Copy only the largest blob pixels from the original mask
        clean_mask[labels == max_label] = mask_array[labels == max_label]
        return clean_mask

    return mask_array

# --- 2. Update Visualization Function to Use It ---
def visualize_prediction(model, dataset, idx=0):
    model.eval()

    # 1. Get data
    image, mask = dataset[idx]
    image_tensor = image.unsqueeze(0).to(DEVICE) # Add batch dim

    # 2. Predict
    with torch.no_grad():
        output = model(image_tensor)
        pred_mask = torch.argmax(output, dim=1).squeeze().cpu().numpy()

    # === INSERTED CLEANING STEP HERE ===
    pred_mask = clean_segmentation(pred_mask)
    # ===================================

    # 3. Prepare for plotting
    # Convert image from Tensor (C,H,W) to Numpy (H,W,C)
    img_np = image.permute(1, 2, 0).cpu().numpy()

    # Normalize image for display (undo the normalization)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # 4. Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_np)
    axes[0].set_title(f"Input Image (Index {idx})")

    axes[1].imshow(mask.cpu().numpy(), cmap='jet', interpolation='nearest')
    axes[1].set_title("Ground Truth (0=BG, 1=Disc, 2=Cup)")

    axes[2].imshow(pred_mask, cmap='jet', interpolation='nearest')
    axes[2].set_title(f"Prediction (Cleaned)")

    plt.show()

# Run it on a random image
idx = random.randint(0, len(dataset)-1)
visualize_prediction(model, dataset, idx)

In [ ]:
# ==========================================
# PLOT TRAINING HISTORY
# ==========================================
import matplotlib.pyplot as plt

# 1. Plot Loss
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss", color='blue', linestyle='-')
plt.plot(history["val_loss"], label="Validation Loss", color='red', linestyle='--')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/figure_training_loss.png', dpi=300)
plt.show()

# 2. Plot Dice Scores
plt.figure(figsize=(8, 5))
plt.plot(history["dice_disc"], label="Optic Disc Dice", color='green', marker='o', markersize=3)
plt.plot(history["dice_cup"], label="Optic Cup Dice", color='orange', marker='s', markersize=3)
plt.xlabel("Epochs")
plt.ylabel("Dice Coefficient")
plt.title("Segmentation Performance (Dice Score)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/figure_dice_score.png', dpi=300)
plt.show()

In [ ]:
# ==========================================
# GENERATE DETAILED RESULTS FOR SECTION 4.5
# ==========================================
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
import seaborn as sns

def generate_paper_results(model, loader, device):
    model.eval()

    all_vcdr_gt = []
    all_vcdr_pred = []

    # Classification Lists (RG vs NRG)
    y_true_class = []
    y_pred_class = []

    # Threshold for Glaucoma (0.65 is standard, adjust if your paper uses 0.5 or 0.7)
    VCDR_THRESHOLD = 0.55

    print("Running evaluation on all images...")

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.cpu().numpy()

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            # Process batch
            for i in range(len(preds)):
                # 1. Clean the mask (Remove noise)
                clean_pred = clean_segmentation(preds[i]) # Using the function we added earlier

                # 2. Calculate vCDR
                gt_vcdr = get_vcdr(masks[i])
                pred_vcdr = get_vcdr(clean_pred)

                all_vcdr_gt.append(gt_vcdr)
                all_vcdr_pred.append(pred_vcdr)

                # 3. Determine Class (RG vs NRG)
                # If vCDR > Threshold -> Glaucoma (1), else Normal (0)
                y_true_class.append(1 if gt_vcdr > VCDR_THRESHOLD else 0)
                y_pred_class.append(1 if pred_vcdr > VCDR_THRESHOLD else 0)

    # --- PLOT 1: SCATTER PLOT ---
    plt.figure(figsize=(6, 6))
    plt.scatter(all_vcdr_gt, all_vcdr_pred, alpha=0.6, color='blue')
    plt.plot([0, 1], [0, 1], 'r--', label="Perfect Agreement")
    plt.xlabel("Ground Truth vCDR")
    plt.ylabel("Predicted vCDR")
    plt.title("vCDR Agreement Analysis")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('/content/drive/MyDrive/figure_vcdr_scatter.png', dpi=300)
    plt.show()

    # --- PLOT 2: CONFUSION MATRIX ---
    cm = confusion_matrix(y_true_class, y_pred_class)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Glaucoma'],
                yticklabels=['Normal', 'Glaucoma'])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Screening Confusion Matrix")
    plt.savefig('/content/drive/MyDrive/figure_confusion_matrix.png', dpi=300)
    plt.show()

    # --- PRINT TABLE METRICS ---
    print("\n" + "="*30)
    print("FINAL METRICS FOR TABLE 4")
    print("="*30)
    print(f"Accuracy:    {accuracy_score(y_true_class, y_pred_class):.4f}")
    print(f"Sensitivity: {recall_score(y_true_class, y_pred_class):.4f}")
    print(f"Specificity: {recall_score(y_true_class, y_pred_class, pos_label=0):.4f}")
    print(f"F1-Score:    {f1_score(y_true_class, y_pred_class):.4f}")
    print("="*30)

# Run it!
# Note: Use val_loader or test_loader here.
generate_paper_results(model, train_loader, DEVICE)

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.interpolate import griddata

def plot_3d_loss_landscape_schematic(history):
    # 1. Prepare Trajectory Data
    epochs = np.array(list(range(1, len(history["train_loss"]) + 1)))
    val_loss = np.array(history["val_loss"])
    train_loss = np.array(history["train_loss"])
    dice = np.array(history["dice_cup"])

    # 2. Generate a "Schematic" Surface (The Mountain)
    # We create a grid around your trajectory to simulate a 'valley'
    # This is for visualization purposes to show convergence.

    # Create a grid of points
    x_grid = np.linspace(min(epochs), max(epochs), 50)
    y_grid = np.linspace(min(val_loss) * 0.8, max(val_loss) * 1.2, 50)
    X, Y = np.meshgrid(x_grid, y_grid)

    # Create a synthetic "Valley" function:
    # Z (Loss) increases as you get further from the ideal Epoch/Loss path
    # This is a mathematical trick to make a pretty surface that respects your data bounds
    Z_approx = np.zeros_like(X)

    # Simple interpolation of your actual train loss to define the "floor" of the valley
    fitted_loss = np.interp(X, epochs, train_loss)

    # Add "walls" to the valley (Loss goes up as validation error deviates)
    # This creates the U-shape curve
    Z = fitted_loss + 0.5 * (Y - np.interp(X, epochs, val_loss))**2

    # 3. Create the Plot
    fig = go.Figure()

    # --- A. The Surface (The Mountain) ---
    fig.add_trace(go.Surface(
        z=Z, x=X, y=Y,
        colorscale='Blues',
        opacity=0.6,
        showscale=False,
        name='Loss Landscape (Schematic)'
    ))

    # --- B. The Trajectory (Your Actual Model) ---
    fig.add_trace(go.Scatter3d(
        x=epochs,
        y=val_loss,
        z=train_loss,
        mode='lines+markers',
        marker=dict(
            size=6,
            color=dice,
            colorscale='Hot', # Fire color for the path
            showscale=True,
            colorbar=dict(title="Dice Score", x=0.9)
        ),
        line=dict(color='red', width=4),
        name='Your Model Path'
    ))

    # --- C. Layout ---
    fig.update_layout(
        title="3D Optimization Landscape (Schematic Visualization)",
        scene=dict(
            xaxis_title='Epochs (Time)',
            yaxis_title='Validation Loss',
            zaxis_title='Training Loss',
            camera=dict(eye=dict(x=1.6, y=1.6, z=1.2))
        ),
        height=800,
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

# Run it
plot_3d_loss_landscape_schematic(history)

In [ ]:
# ===============================
# HELPER: CALCULATE vCDR
# ===============================
import numpy as np

def get_vcdr(pred_mask):
    """
    Computes Vertical Cup-to-Disc Ratio (vCDR).
    pred_mask: 2D numpy array where 0=BG, 1=Disc, 2=Cup
    """
    # 1. Find rows containing Disc (1) or Cup (2)
    disc_rows = np.where(np.any((pred_mask == 1) | (pred_mask == 2), axis=1))[0]

    # 2. Find rows containing Cup (2)
    cup_rows = np.where(np.any(pred_mask == 2, axis=1))[0]

    # 3. Handle cases where Disc or Cup is missing
    if len(disc_rows) == 0:
        return 0.0 # No disc found
    if len(cup_rows) == 0:
        return 0.0 # Disc found, but no cup found

    # 4. Calculate Vertical Diameters
    v_disc = disc_rows.max() - disc_rows.min()
    v_cup  = cup_rows.max()  - cup_rows.min()

    # 5. Return Ratio
    if v_disc == 0:
        return 0.0

    return v_cup / v_disc

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "epoch": epoch,
}, "unet_glaucoma.pth")


In [ ]:
model = UNet(n_classes=3).to(DEVICE)
checkpoint = torch.load("unet_glaucoma.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])


In [ ]:
# ===============================
# SAVE MODEL (Corrected)
# ===============================
import os

# 1. Define the folder (Directory)
SAVE_DIR = '/content/'

# 2. Make sure the folder exists
os.makedirs(SAVE_DIR, exist_ok=True)

# 3. Define the full file path (Folder + Filename)
SAVE_PATH = os.path.join(SAVE_DIR, 'glaucoma_unet_f.pth')

# 4. Save
torch.save(model.state_dict(), SAVE_PATH)
print(f"✅ Model saved successfully to: {SAVE_PATH}")